In [17]:
import truststore
truststore.inject_into_ssl()

In [18]:
from dotenv import load_dotenv
load_dotenv()

True

In [19]:
# INDEX

import bs4
from langchain_community.document_loaders import WebBaseLoader
loader = WebBaseLoader(
    web_paths=("https://lilianweng.github.io/posts/2023-06-23-agent/",),
    bs_kwargs=dict(
        parse_only=bs4.SoupStrainer(
            class_=("post-content", "post-title", "post-header")
        )
    ),
)
blog_docs = loader.load()

# Split
from langchain.text_splitter import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=300, 
    chunk_overlap=50)

# Make splits
splits = text_splitter.split_documents(blog_docs)

# Index
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
vectorstore = Chroma.from_documents(documents=splits, 
                                    embedding=HuggingFaceEmbeddings())

retriever = vectorstore.as_retriever()

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4654.10it/s]


In [20]:
# Prompt

from langchain_core.prompts import ChatPromptTemplate

# Multi Query: Different Perspectives
template = """You are an AI language model assistant. Your task is to generate five 
different versions of the given user question to retrieve relevant documents from a vector 
database. By generating multiple perspectives on the user question, your goal is to help
the user overcome some of the limitations of the distance-based similarity search. 
Provide these alternative questions separated by newlines. Original question: {question}"""
prompt_perspectives = ChatPromptTemplate.from_template(template)

from langchain_core.output_parsers import StrOutputParser
from langchain_groq import ChatGroq

generate_queries = (
    prompt_perspectives 
    | ChatGroq(model="llama-3.3-70b-versatile", temperature=0) 
    | StrOutputParser() 
    | (lambda x: x.split("\n"))
)

In [21]:
from langchain.load import dumps, loads

def get_unique_union(documents: list[list]):
    """ Unique union of retrieved docs """
    # Flatten list of lists, and convert each Document to string
    flattened_docs = [dumps(doc) for sublist in documents for doc in sublist]
    # Get unique documents
    unique_docs = list(set(flattened_docs))
    # Return
    return [loads(doc) for doc in unique_docs]

# Retrieve
question = "What is task decomposition for LLM agents?"
retrieval_chain = generate_queries | retriever.map() | get_unique_union
docs = retrieval_chain.invoke({"question":question})
len(docs)

/var/folders/_d/tj_f_hcs5gd3hx64vjm05drw0000gp/T/ipykernel_16035/2784938305.py:10: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit list of allowed classes (or 'messages' for untrusted input that contains only chat messages) to suppress this warning.
  return [loads(doc) for doc in unique_docs]


6

In [22]:
question = "What is task decomposition for LLM agents?"
llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)

o1 = prompt_perspectives.invoke({"question": question})
o2 = llm.invoke(o1)
o3 =  StrOutputParser().invoke(o2) 
o4 = o3.split("\n")

print(f"o1: {o1}")
print(f"o2: {o2}")
print(f"o3: {o3}")
print(f"o4: {o4}")


o5 = retriever.map().invoke(o4)
print(f"o5: {o5}")

o6 = get_unique_union(o5)
print(f"o6: {o6}")

o1: messages=[HumanMessage(content='You are an AI language model assistant. Your task is to generate five \ndifferent versions of the given user question to retrieve relevant documents from a vector \ndatabase. By generating multiple perspectives on the user question, your goal is to help\nthe user overcome some of the limitations of the distance-based similarity search. \nProvide these alternative questions separated by newlines. Original question: What is task decomposition for LLM agents?', additional_kwargs={}, response_metadata={})]
o2: content='What is the process of breaking down tasks for large language model agents to improve their performance and efficiency?\n\nHow do LLM agents utilize task decomposition to enhance their ability to understand and complete complex tasks?\n\nWhat role does task decomposition play in the development and training of large language model agents, and what benefits does it provide?\n\nCan you explain the concept of task decomposition in the context

/var/folders/_d/tj_f_hcs5gd3hx64vjm05drw0000gp/T/ipykernel_16035/2784938305.py:10: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit list of allowed classes (or 'messages' for untrusted input that contains only chat messages) to suppress this warning.
  return [loads(doc) for doc in unique_docs]


In [23]:
from operator import itemgetter
from langchain_groq import ChatGroq

# RAG
template = """Answer the following question based on this context:

{context}

Question: {question}
"""

prompt = ChatPromptTemplate.from_template(template)

llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)

final_rag_chain = (
    {"context": retrieval_chain, 
     "question": itemgetter("question")} 
    | prompt
    | llm
    | StrOutputParser()
)

final_rag_chain.invoke({"question":question})

/var/folders/_d/tj_f_hcs5gd3hx64vjm05drw0000gp/T/ipykernel_16035/2784938305.py:10: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit list of allowed classes (or 'messages' for untrusted input that contains only chat messages) to suppress this warning.
  return [loads(doc) for doc in unique_docs]


'Task decomposition for LLM (Large Language Model) agents refers to the process of breaking down complex tasks into smaller, more manageable subgoals or steps. This allows the agent to handle complex tasks more efficiently. Task decomposition can be achieved through various methods, including:\n\n1. Simple prompting: Using prompts like "Steps for XYZ" or "What are the subgoals for achieving XYZ?" to guide the LLM in decomposing the task.\n2. Task-specific instructions: Providing specific instructions tailored to the task at hand, such as "Write a story outline" for writing a novel.\n3. Human inputs: Incorporating human feedback or guidance to help the LLM decompose the task.\n4. Chain of Thought (CoT): Instructing the model to "think step by step" to utilize more test-time computation and decompose hard tasks into smaller steps.\n5. Tree of Thoughts: Exploring multiple reasoning possibilities at each step, creating a tree structure, and using techniques like breadth-first search (BFS) 

In [24]:
# Adding ranking to retival documents

from langchain.load import dumps, loads

def reciprocal_rank_fusion(results: list[list], k=60):
    fused_scores = {}

    for docs in results:
        # Iterate through each document in the list, with its rank (position in the list)
        for rank, doc in enumerate(docs):
            # Convert the document to a string format to use as a key (assumes documents can be serialized to JSON)
            doc_str = dumps(doc)
            # If the document is not yet in the fused_scores dictionary, add it with an initial score of 0
            if doc_str not in fused_scores:
                fused_scores[doc_str] = 0
            # Retrieve the current score of the document, if any
            previous_score = fused_scores[doc_str]
            # Update the score of the document using the RRF formula: 1 / (rank + k)
            fused_scores[doc_str] += 1 / (rank + k)

    # Sort the documents based on their fused scores in descending order to get the final reranked results
    reranked_results = [
        (loads(doc), score)
        for doc, score in sorted(fused_scores.items(), key=lambda x: x[1], reverse=True)
    ]

    # Return the reranked results as a list of tuples, each containing the document and its fused score
    return reranked_results

retrieval_chain_rag_fusion = generate_queries | retriever.map() | reciprocal_rank_fusion
docs = retrieval_chain_rag_fusion.invoke({"question": question})
len(docs)

/var/folders/_d/tj_f_hcs5gd3hx64vjm05drw0000gp/T/ipykernel_16035/3253735109.py:23: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit list of allowed classes (or 'messages' for untrusted input that contains only chat messages) to suppress this warning.
  (loads(doc), score)


6

In [16]:
template = """Answer the following question based on this context:

{context}

Question: {question}
"""

prompt = ChatPromptTemplate.from_template(template)

final_rag_chain = (
    {"context": retrieval_chain_rag_fusion, 
     "question": itemgetter("question")} 
    | prompt
    | llm
    | StrOutputParser()
)

final_rag_chain.invoke({"question":question})

/var/folders/_d/tj_f_hcs5gd3hx64vjm05drw0000gp/T/ipykernel_16035/3253735109.py:23: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit list of allowed classes (or 'messages' for untrusted input that contains only chat messages) to suppress this warning.
  (loads(doc), score)


'Task decomposition for LLM (Large Language Model) agents refers to the process of breaking down complex tasks into smaller, more manageable subgoals. This allows the agent to handle complex tasks more efficiently. Task decomposition can be done in several ways, including:\n\n1. Using simple prompting, such as "Steps for XYZ" or "What are the subgoals for achieving XYZ?"\n2. Using task-specific instructions, such as "Write a story outline" for writing a novel\n3. With human inputs\n\nAdditionally, techniques like Chain of Thought (CoT) and Tree of Thoughts can be used to decompose tasks into smaller steps and explore multiple reasoning possibilities at each step. This can help the agent to better understand the task and generate more effective solutions.'

Decomposition

In [26]:
from langchain_core.prompts import ChatPromptTemplate

# Decomposition
template = """You are a helpful assistant that generates multiple sub-questions related to an input question. \n
The goal is to break down the input into a set of sub-problems / sub-questions that can be answers in isolation. \n
Generate multiple search queries related to: {question} \n
Output (3 queries):"""
prompt_decomposition = ChatPromptTemplate.from_template(template)

In [27]:
from langchain_groq import ChatGroq
from langchain_core.output_parsers import StrOutputParser

# LLM
llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)

# Chain
generate_queries_decomposition = ( prompt_decomposition | llm | StrOutputParser() | (lambda x: x.split("\n")))

# Run
question = "What are the main components of an LLM-powered autonomous agent system?"
questions = generate_queries_decomposition.invoke({"question":question})

In [28]:
template = """Here is the question you need to answer:

\n --- \n {question} \n --- \n

Here is any available background question + answer pairs:

\n --- \n {q_a_pairs} \n --- \n

Here is additional context relevant to the question: 

\n --- \n {context} \n --- \n

Use the above context and any background question + answer pairs to answer the question: \n {question}
"""

decomposition_prompt = ChatPromptTemplate.from_template(template)

In [30]:
from operator import itemgetter
from langchain_core.output_parsers import StrOutputParser

def format_qa_pair(question, answer):
    """Format Q and A pair"""
    
    formatted_string = ""
    formatted_string += f"Question: {question}\nAnswer: {answer}\n\n"
    return formatted_string.strip()

# llm
llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)

q_a_pairs = ""
for q in questions:
    
    rag_chain = (
    {"context": itemgetter("question") | retriever, 
     "question": itemgetter("question"),
     "q_a_pairs": itemgetter("q_a_pairs")} 
    | decomposition_prompt
    | llm
    | StrOutputParser())

    answer = rag_chain.invoke({"question":q,"q_a_pairs":q_a_pairs})
    print(f"<<<<<<<<<<<<. q: {q},           q_a_pairs: {q_a_pairs}")
    q_a_pair = format_qa_pair(q,answer)
    q_a_pairs = q_a_pairs + "\n---\n"+  q_a_pair

<<<<<<<<<<<<. q: To break down the question into manageable sub-questions, here are three search queries:,           q_a_pairs: 
<<<<<<<<<<<<. q: ,           q_a_pairs: 
---
Question: To break down the question into manageable sub-questions, here are three search queries:
Answer: Unfortunately, you didn't provide the actual question that needs to be broken down into manageable sub-questions. However, I can provide a general framework for breaking down a question into sub-questions using search queries.

To break down a question into manageable sub-questions, you can use the following steps:

1. Identify the key components of the question: What are the main topics, concepts, or keywords involved in the question?
2. Determine the scope of the question: What is the context, timeframe, or geographical area relevant to the question?
3. Formulate search queries: Based on the key components and scope, create search queries that can help you find relevant information.

Here are three general s

In [33]:
from langchain import hub

# RAG prompt
prompt_rag = ChatPromptTemplate.from_template("""Answer the question based only on the following context:

{context}

Question: {question}
""")

def retrieve_and_rag(question,prompt_rag,sub_question_generator_chain):
    """RAG on each sub-question"""
    
    # Use our decomposition / 
    sub_questions = sub_question_generator_chain.invoke({"question":question})
    
    # Initialize a list to hold RAG chain results
    rag_results = []
    
    for sub_question in sub_questions:
        
        # Retrieve documents for each sub-question
        retrieved_docs = retriever
        
        # Use retrieved documents and sub-question in RAG chain
        answer = (prompt_rag | llm | StrOutputParser()).invoke({"context": retrieved_docs, 
                                                                "question": sub_question})
        rag_results.append(answer)
    
    return rag_results,sub_questions

# Wrap the retrieval and RAG process in a RunnableLambda for integration into a chain
answers, questions = retrieve_and_rag(question, prompt_rag, generate_queries_decomposition)

In [34]:
def format_qa_pairs(questions, answers):
    """Format Q and A pairs"""
    
    formatted_string = ""
    for i, (question, answer) in enumerate(zip(questions, answers), start=1):
        formatted_string += f"Question {i}: {question}\nAnswer {i}: {answer}\n\n"
    return formatted_string.strip()

context = format_qa_pairs(questions, answers)

# Prompt
template = """Here is a set of Q+A pairs:

{context}

Use these to synthesize an answer to the question: {question}
"""

prompt = ChatPromptTemplate.from_template(template)

final_rag_chain = (
    prompt
    | llm
    | StrOutputParser()
)

final_rag_chain.invoke({"context":context,"question":question})

"Based on the provided context, it seems that the main components of an LLM-powered autonomous agent system are not explicitly mentioned. However, we can infer some possible components based on the questions and answers provided.\n\nAn LLM-powered autonomous agent system likely consists of several key components, including:\n\n1. **Large Language Model (LLM)**: This is the core component that enables the autonomous agent to understand and generate human-like language.\n2. **Perception System**: This component is responsible for perceiving the environment and gathering data, which is then used by the LLM to make decisions.\n3. **Action System**: This component is responsible for taking actions in the environment based on the decisions made by the LLM.\n4. **Decision-Making and Planning Algorithms**: These algorithms are used to make informed decisions and plan actions, and are likely integrated with the LLM to enable the autonomous agent to navigate through tasks.\n5. **Vector Store**: 

Step Back

In [36]:
from langchain_core.prompts import ChatPromptTemplate, FewShotChatMessagePromptTemplate
examples = [
    {
        "input": "Could the members of The Police perform lawful arrests?",
        "output": "what can the members of The Police do?",
    },
    {
        "input": "Jan Sindel’s was born in what country?",
        "output": "what is Jan Sindel’s personal history?",
    },
]
# We now transform these to example messages
example_prompt = ChatPromptTemplate.from_messages(
    [
        ("human", "{input}"),
        ("ai", "{output}"),
    ]
)
few_shot_prompt = FewShotChatMessagePromptTemplate(
    example_prompt=example_prompt,
    examples=examples,
)
prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """You are an expert at world knowledge. Your task is to step back and paraphrase a question to a more generic step-back question, which is easier to answer. Here are a few examples:""",
        ),
        # Few shot examples
        few_shot_prompt,
        # New question
        ("user", "{question}"),
    ]
)

In [37]:
generate_queries_step_back = prompt | ChatGroq(model="llama-3.3-70b-versatile", temperature=0) | StrOutputParser()
question = "What is task decomposition for LLM agents?"
generate_queries_step_back.invoke({"question": question})

'How do complex tasks get broken down into simpler ones?'

In [ ]:
from langchain_core.runnables import RunnableLambda

response_prompt_template = """You are an expert of world knowledge. I am going to ask you a question. Your response should be comprehensive and not contradicted with the following context if they are relevant. Otherwise, ignore them if they are not relevant.

# {normal_context}
# {step_back_context}

# Original Question: {question}
# Answer:"""
response_prompt = ChatPromptTemplate.from_template(response_prompt_template)

chain = (
    {
        # Retrieve context using the normal question
        "normal_context": RunnableLambda(lambda x: x["question"]) | retriever,
        # Retrieve context using the step-back question
        "step_back_context": generate_queries_step_back | retriever,
        # Pass on the question
        "question": lambda x: x["question"],
    }
    | response_prompt
    | ChatGroq(model="llama-3.3-70b-versatile", temperature=0)
    | StrOutputParser()
)

chain.invoke({"question": question})

'Task decomposition for LLM (Large Language Model) agents refers to the process of breaking down complex tasks into smaller, more manageable sub-tasks or steps. This is a crucial component of planning and problem-solving for LLM agents, as it enables them to tackle intricate tasks by dividing them into simpler, more manageable parts.\n\nThere are several techniques used for task decomposition in LLM agents, including:\n\n1. **Chain of Thought (CoT)**: This involves instructing the model to "think step by step" to utilize more test-time computation to decompose hard tasks into smaller and simpler steps. CoT transforms big tasks into multiple manageable tasks and provides insight into the model\'s thinking process.\n2. **Tree of Thoughts**: This extends CoT by exploring multiple reasoning possibilities at each step. It first decomposes the problem into multiple thought steps and generates multiple thoughts per step, creating a tree structure. The search process can be performed using bre

HyDE

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

# HyDE document generation
template = """Please write a scientific paper passage to answer the question
Question: {question}
Passage:"""
prompt_hyde = ChatPromptTemplate.from_template(template)

generate_docs_for_retrieval = (
    prompt_hyde | ChatGroq(model="llama-3.3-70b-versatile", temperature=0) | StrOutputParser() 
)

# Run
question = "What is task decomposition for LLM agents?"
generate_docs_for_retrieval.invoke({"question":question})

'**Task Decomposition for Large Language Model (LLM) Agents: A Framework for Efficient Problem-Solving**\n\nTask decomposition is a crucial technique employed by Large Language Model (LLM) agents to efficiently solve complex problems. At its core, task decomposition involves breaking down a complex task into a series of smaller, more manageable sub-tasks that can be executed sequentially or in parallel. This approach enables LLM agents to tackle intricate problems by dividing them into simpler, more tractable components, thereby reducing the computational complexity and improving overall performance.\n\nIn the context of LLM agents, task decomposition typically involves a hierarchical representation of the problem-solving process. The agent begins by analyzing the input task and identifying the key objectives, constraints, and requirements. The task is then decomposed into a set of sub-tasks, each of which is associated with a specific goal or objective. These sub-tasks are further ref

In [43]:
retrieval_chain = generate_docs_for_retrieval | retriever 
retrieved_docs = retrieval_chain.invoke({"question":question})
retrieved_docs

[Document(metadata={'source': 'https://lilianweng.github.io/posts/2023-06-23-agent/'}, page_content='Component One: Planning#\nA complicated task usually involves many steps. An agent needs to know what they are and plan ahead.\nTask Decomposition#\nChain of thought (CoT; Wei et al. 2022) has become a standard prompting technique for enhancing model performance on complex tasks. The model is instructed to “think step by step” to utilize more test-time computation to decompose hard tasks into smaller and simpler steps. CoT transforms big tasks into multiple manageable tasks and shed lights into an interpretation of the model’s thinking process.\nTree of Thoughts (Yao et al. 2023) extends CoT by exploring multiple reasoning possibilities at each step. It first decomposes the problem into multiple thought steps and generates multiple thoughts per step, creating a tree structure. The search process can be BFS (breadth-first search) or DFS (depth-first search) with each state evaluated by a

In [44]:
template = """Answer the following question based on this context:

{context}

Question: {question}
"""

prompt = ChatPromptTemplate.from_template(template)

final_rag_chain = (
    prompt
    | llm
    | StrOutputParser()
)

final_rag_chain.invoke({"context":retrieved_docs,"question":question})

'Task decomposition for LLM (Large Language Model) agents refers to the process of breaking down complex tasks into smaller, more manageable subgoals or steps. This allows the agent to efficiently handle complex tasks by decomposing them into simpler ones. Task decomposition can be achieved through various methods, including:\n\n1. Using simple prompting techniques, such as "Steps for XYZ. 1.", "What are the subgoals for achieving XYZ?"\n2. Utilizing task-specific instructions, like "Write a story outline." for writing a novel\n3. Incorporating human inputs\n\nAdditionally, techniques like Chain of Thought (CoT) and Tree of Thoughts can be employed to facilitate task decomposition. CoT involves instructing the model to "think step by step" to decompose hard tasks into smaller steps, while Tree of Thoughts extends CoT by exploring multiple reasoning possibilities at each step, creating a tree structure.'